# 分类资金流：来源、历史范围与版本资格

固定只读请求；不将分类成交额冒充ETF申赎或已认证主动订单流。日频经济日期不是信息可用时点。

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import os,json,hashlib,time
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from futu import OpenQuoteContext,PeriodType,RET_OK,SysConfig
from src.data.minute_intake import atomic_json
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():
 if root==root.parent:raise RuntimeError('repository directory required')
 root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/capital_flow_source_20261014';plan=json.loads((card/'plan.json').read_text());out=root/plan['raw_directory'];out.mkdir(exist_ok=True,parents=True);raw={};receipts=[];legacy={}
for name,digest in plan['legacy_hashes'].items():
 p=root/'data/raw/external'/name;assert hashlib.sha256(p.read_bytes()).hexdigest()==digest;legacy[name]=pd.read_csv(p)
SysConfig.set_all_thread_daemon(True);context=OpenQuoteContext(host='127.0.0.1',port=11111)
try:
 for symbol in plan['symbols']:
  for request in plan['requests']+[{'key':'intraday','start':None,'end':None}]:
   key=symbol+'_'+request['key'];rp=out/(key+'.json');path=out/(key+'.parquet')
   if rp.exists():
    receipt=json.loads(rp.read_text())
    if receipt['ret_code']==RET_OK:assert hashlib.sha256(path.read_bytes()).hexdigest()==receipt['sha256'];raw[key]=pd.read_parquet(path)
    receipts.append(receipt);continue
   started=datetime.now(timezone.utc).isoformat();time.sleep(1.)
   ret,data=context.get_capital_flow('US.'+symbol,period_type=PeriodType.INTRADAY if request['key']=='intraday' else PeriodType.DAY,start=request['start'],end=request['end']);received=datetime.now(timezone.utc).isoformat();receipt={'key':key,'symbol':symbol,'request':request,'ret_code':int(ret),'started_at_utc':started,'received_at_utc':received,'read_only':True}
   if ret==RET_OK:
    data.to_parquet(path,index=False);raw[key]=data;receipt.update(path=path.name,sha256=hashlib.sha256(path.read_bytes()).hexdigest(),rows=len(data))
   else:receipt['error']=str(data)[:500]
   atomic_json(receipt,rp);receipts.append(receipt)
finally:context.close()
fields=['in_flow','super_in_flow','big_in_flow','mid_in_flow','sml_in_flow','main_in_flow'];source_rows=[];frames=[];intra=[];comparisons=[]
all_sources={**{'legacy_'+name:d for name,d in legacy.items()},**raw}
for key,d in all_sources.items():
 t=pd.to_datetime(d.capital_flow_item_time);is_intraday='intraday' in key;values=d[fields].apply(pd.to_numeric,errors='coerce');row={'source':key,'kind':'intraday' if is_intraday else 'daily','rows':len(d),'dates':t.dt.normalize().nunique(),'first':str(t.min()),'last':str(t.max()),'duplicates':int(t.duplicated().sum()),'last_valid_time_examples':'|'.join(d.last_valid_time.astype(str).unique()[:3])}
 row['max_total_components_difference']=float((values.in_flow-values[['super_in_flow','big_in_flow','mid_in_flow','sml_in_flow']].sum(axis=1,min_count=4)).abs().max());row['max_main_components_difference']=float((values.main_in_flow-values.super_in_flow-values.big_in_flow).abs().max());source_rows.append(row);assert not t.duplicated().any()
 if is_intraday:intra.append(dict(source=key,first_flow=float(values.in_flow.iloc[0]),last_flow=float(values.in_flow.iloc[-1]),sum_reported_points=float(values.in_flow.sum()),last_date=str(t.max().date())));continue
 instrument=key.replace('legacy_futu_capital_flow_day_US_','LEGACY_').replace('.csv','').upper();f=values.copy();f['datetime']=t;f['instrument']=instrument;frames.append(f)
# Compare only overlapping daily observations, retaining every vintage separately.
for symbol in plan['symbols']:
 old=legacy['futu_capital_flow_day_US_'+symbol+'.csv'].copy();old.index=pd.to_datetime(old.capital_flow_item_time)
 for request in plan['requests']:
  key=symbol+'_'+request['key']
  if key not in raw:continue
  new=raw[key].copy();new.index=pd.to_datetime(new.capital_flow_item_time);common=old.index.intersection(new.index)
  for field in fields:
   a=pd.to_numeric(old.loc[common,field],errors='coerce');b=pd.to_numeric(new.loc[common,field],errors='coerce');diff=b-a;comparisons.append({'symbol':symbol,'request':request['key'],'field':field,'overlap_dates':len(common),'known_pairs':int(diff.notna().sum()),'changed_pairs_above_1cent':int(diff.abs().gt(.01).sum()),'max_absolute_difference':float(diff.abs().max())})
for row in intra:
 symbol='SOXX' if 'SOXX' in row['source'] else 'SOXL';key='legacy_futu_capital_flow_day_US_'+symbol+'.csv' if row['source'].startswith('legacy_') else symbol+'_recent';d=all_sources.get(key)
 if d is not None:
  same=d.loc[pd.to_datetime(d.capital_flow_item_time).dt.strftime('%Y-%m-%d').eq(row['last_date']),'in_flow'];row['matching_daily_flow']=float(pd.to_numeric(same).iloc[0]) if len(same)==1 else np.nan;row['endpoint_minus_daily']=row['last_flow']-row['matching_daily_flow'];row['sum_minus_daily']=row['sum_reported_points']-row['matching_daily_flow']
frame=pd.concat(frames,ignore_index=True);provider=root/plan['provider'];manifest=build_qlib_store_from_frame(frame,provider_uri=provider,freq='day',as_of=plan['as_of'],field_map={f:f for f in fields},source='Futu classified trading-flow source-vintage observations; economic dates only, NOT decision-available timestamps, not ETF creations/redemptions')
qlib.init(provider_uri=str(provider),region='us');native=load_qlib_feature_frame(QlibProviderConfig(provider,freq='day'),instruments=frame.instrument.unique().tolist(),fields=['$'+f for f in fields],start=str(frame.datetime.min().date()),end=plan['as_of']);expected=frame.set_index(['instrument','datetime'])
for f in fields:np.testing.assert_allclose(native['$'+f].reindex(expected.index),expected[f],rtol=2e-7,atol=.01,equal_nan=True)
summary={'experiment':card.name,'decision':'source_scope_and_vintage_audit_only','new_fits':0,'new_backtests':0,'future_outcome_tests':0,'objective_achieved':False,'forecast_input_qualified':False,'historical_first_receipt_certified':False,'request_count':len(receipts),'successful_requests':sum(r['ret_code']==RET_OK for r in receipts),'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'scope':plan['scope'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='read_only_vendor_flow_qualification',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0)
 rec.save_objects(artifact_path='inputs',plan=plan,receipts=receipts,raw_sources=raw,legacy_sources=legacy,provider_manifest=manifest)
 rec.save_objects(artifact_path='analysis',native=native,source_coverage=pd.DataFrame(source_rows),vintage_comparison=pd.DataFrame(comparisons),intraday_semantics=pd.DataFrame(intra),summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(source_rows).to_csv(card/'source_coverage.csv',index=False);pd.DataFrame(comparisons).to_csv(card/'vintage_comparison.csv',index=False);pd.DataFrame(intra).to_csv(card/'intraday_semantics.csv',index=False);atomic_json(receipts,card/'request_receipts.json')
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False));assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print(pd.DataFrame(source_rows).to_string(index=False));print(pd.DataFrame(comparisons).to_string(index=False));print(pd.DataFrame(intra).to_string(index=False));print([{k:v for k,v in r.items() if k in ['key','ret_code','rows','error']} for r in receipts])


# 分类资金流来源可取得，但历史时点与完整日内覆盖仍不足

本轮沿用现有Futu只读权限，0训练、0回测、0收益筛选、0订单。原始响应与旧CSV分别保存，建立Qlib日频来源版本store并原生读回；经济日期只表示观察所属日，不作为历史决策可用时间。生产release未改，稳定净超额目标未完成。

固定八个请求：SOXX/SOXL各请求2024全年、2025全年、2025-10-14至2026-10-13以及当前日内。八个请求均返回成功码，但2024两次为空；2025各只有55行，实际从2025-10-14开始。最近一年各251行，覆盖2025-10-14至2026-10-13。本次观察显示历史窗口受限，不能把成功码或请求日期当成实际覆盖，也不据此断言所有其他接口都不可能提供更早数据。

旧文件各有251个日频日期（2025-05-23至2026-05-22），新旧最近一年响应各重叠153天，六字段逐值相同，最大差0。保留两个版本后各有349个不同日期；不覆盖旧快照，也不将55行子请求的重复日期算成新样本。这是两次快照间一致性的证据，不证明更早版本从未修订或当时已经公布。

已有日内文件仅2026-05-22，本次取得2026-10-13；每只各391个点，从09:30的0到16:00。四份快照末值均精确等于匹配日频in_flow，而逐点求和不等于日频值。例如2026-10-13 SOXX末值约-2694万美元，逐点和约-277.76亿美元。四个观测序列与累计量语义一致；不能把每个点当独立分钟流量相加。若未来构造区间变化，应先验证同一交易日、实际信息时点及累计量重置，再使用差分，不能跨日直接差分。

字段结构也高度冗余：in_flow等于四种规模分类之和，日频main_in_flow等于super_in_flow+big_in_flow，差仅浮点精度量级。不能把总量、主力、全部子项当成相互独立的多个新信息源。它们是供应商分类的成交资金统计，不是ETF一级市场净申赎，也尚未认证为真实主动订单方向或盘口不平衡。

日频last_valid_time为N/A；日内last_valid_time为供应商最后有效时间，不能替代逐点首次接收时间。本次明确保存请求开始和响应接收UTC时间；旧文件的首次采集时间未补造。日内字段的时间说明是分钟开始时间，不能不经验证就套用已有K线右端闭合合同。

决定：source_scope_and_vintage_audit_only，forecast_input_qualified=false。这些数据可能提供不同于OHLCV汇总的分类成交信息，但现在缺少合格历史分钟序列、当时可用版本及分类规则的进一步确认；没有检验任何收益关系，也没有发现可宣称的净超额。不能把日频全年统计放进当天早盘，更不能把当前日内快照回填历史。

下一步是一次固定决策窗口的只读接收时点核查：保存原始快照、请求/响应时间、供应商时间和数据所属交易日，区分已收到的当前信息与旧日回放；过期或迟到快照保留未知。它只能建立前向数据资格，单日采集不能成为稳定超额证据。历史信息检验仍需更长且有版本/时点证据的来源，不以滞后天数搜索替代这些证据。

依据：[Futu官方资金流接口](https://openapi.futunn.com/futu-api-doc/en/quote/get-capital-flow.html)的字段定义及本机SDK接口。官方页面直接抓取曾超时，官方搜索索引可读；历史实际范围以上述真实响应为准。

[实际覆盖](source_coverage.csv) · [快照对照](vintage_comparison.csv) · [累计量检查](intraday_semantics.csv) · [请求回执](request_receipts.json)

Recorder `c0e103ed01cf44a7887ec7e52dfeb6c1`。原始数据及大证据归档NAS。
